# 05 · Datos para Power BI (modelo estrella)

Power BI ya puede leer los CSV de los notebooks anteriores, pero para un dashboard ordenado conviene un
**modelo estrella**: las **tablas de hechos** (lo que pasó) se conectan a **tablas de dimensiones**
(quién, cuándo, en qué turno). Así **un solo filtro** de máquina, turno o fecha afecta a todas las páginas.

```
                    dim_fechas
                        │
 dim_maquinas ── oee_turnos · alertas_detalle · alertas_por_turno · predicciones_fallas ── dim_turnos
```

En este notebook se generan **4 CSV nuevos** (los existentes no cambian):

| CSV | Tipo | Para qué |
|---|---|---|
| `dim_maquinas.csv` | Dimensión | Nombre para mostrar de cada máquina |
| `dim_turnos.csv` | Dimensión | Ordenar los turnos Mañana → Tarde → Noche (no alfabético) |
| `dim_fechas.csv` | Dimensión | Calendario en español: mes, semana, día de la semana |
| `alertas_detalle.csv` | Hechos | Una fila por **señal** de cada alerta, para graficar causas y acciones |

> Usa los CSV generados por los notebooks 01, 02 y 03.

## 0. Preparación

In [1]:
import pandas as pd

from planta import config
from planta.data.dimensiones import crear_dim_fechas_desde_datos, crear_dim_maquinas, crear_dim_turnos
from planta.data.guardado import guardar_csv_procesado
from planta.models.alertas import desglosar_senales, preparar_ciclos_en_riesgo

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

produccion = pd.read_csv(config.RUTA_CSV_PRODUCCION)
predicciones = pd.read_csv(config.RUTA_CSV_PREDICCIONES)
oee = pd.read_csv(config.RUTA_CSV_OEE, parse_dates=["fecha_turno"])
alertas_por_turno = pd.read_csv(config.RUTA_CSV_ALERTAS, parse_dates=["fecha_turno"])

## 1. Dimensiones

### Máquinas
Se leen de `config.py` (`CANTIDAD_MAQUINAS`): si la planta creciera, no hay que tocar este notebook.

In [2]:
dim_maquinas = crear_dim_maquinas()
dim_maquinas

,id_maquina,nombre_maquina
0,M01,Máquina 01
1,M02,Máquina 02
2,M03,Máquina 03
3,M04,Máquina 04
4,M05,Máquina 05


### Turnos
`orden_turno` sirve para la opción **Ordenar por columna** de Power BI: sin ella, los turnos saldrían
en orden alfabético (Mañana, Noche, Tarde).

In [3]:
dim_turnos = crear_dim_turnos()
dim_turnos

,turno,orden_turno,hora_inicio,hora_fin,horario
0,Mañana,1,6,14,06:00-14:00
1,Tarde,2,14,22,14:00-22:00
2,Noche,3,22,6,22:00-06:00


### Calendario
Un día por fila, desde la primera hasta la última **fecha de jornada** del OEE.
Los nombres de meses y días están escritos en el código para que salgan en español en cualquier computadora.

In [4]:
dim_fechas = crear_dim_fechas_desde_datos(oee["fecha_turno"])
print(f"{len(dim_fechas)} días: del {dim_fechas['fecha'].min():%Y-%m-%d} al {dim_fechas['fecha'].max():%Y-%m-%d}")
dim_fechas.head(8)

43 días: del 2026-09-01 al 2026-10-13


,fecha,anio,mes_numero,mes_nombre,semana_iso,dia_semana_numero,dia_semana_nombre,es_fin_de_semana
0,2026-09-01,2026,9,Septiembre,36,2,Martes,0
1,2026-09-02,2026,9,Septiembre,36,3,Miércoles,0
2,2026-09-03,2026,9,Septiembre,36,4,Jueves,0
3,2026-09-04,2026,9,Septiembre,36,5,Viernes,0
4,2026-09-05,2026,9,Septiembre,36,6,Sábado,1
5,2026-09-06,2026,9,Septiembre,36,7,Domingo,1
6,2026-09-07,2026,9,Septiembre,37,1,Lunes,0
7,2026-09-08,2026,9,Septiembre,37,2,Martes,0


## 2. Detalle de alertas (una señal por fila)

En el correo, una alerta con dos causas las muestra juntas en una celda (*"Sobreesfuerzo; Herramienta desgastada"*).
Power BI no puede contar causas así, por eso aquí cada **señal** ocupa su propia fila.

> **Importante:** como una alerta puede ocupar varias filas, en Power BI las alertas se cuentan con
> `DISTINCTCOUNT(id_registro)`, nunca contando filas.

In [5]:
ciclos_en_riesgo = preparar_ciclos_en_riesgo(produccion, predicciones)
alertas_detalle = desglosar_senales(ciclos_en_riesgo)

print(f"Filas (señales): {len(alertas_detalle)}")
print(f"Alertas distintas: {alertas_detalle['id_registro'].nunique()}")
alertas_detalle.head(6)

Filas (señales): 377
Alertas distintas: 284


,id_registro,id_maquina,fecha_hora,fecha_turno,turno,probabilidad_falla,tipo_senal,senal,accion_sugerida
0,4049,M03,2026-09-01 12:30:00,2026-09-01,Mañana,0.9800,Potencia baja,Potencia baja (3 410 W),Revisar parámetros de corte
1,2016,M02,2026-09-01 15:00:00,2026-09-01,Tarde,0.9233,Potencia alta,Potencia alta (9 033 W),Revisar parámetros de corte
2,6025,M04,2026-09-01 18:00:00,2026-09-01,Tarde,0.8867,Potencia alta,Potencia alta (9 273 W),Revisar parámetros de corte
3,4072,M03,2026-09-02 00:00:00,2026-09-01,Noche,0.8533,Potencia alta,Potencia alta (9 562 W),Revisar parámetros de corte
4,4079,M03,2026-09-02 03:30:00,2026-09-01,Noche,0.7400,Combinación de variables fuera de lo normal,Combinación de variables fuera de lo normal,Inspección general de la máquina
5,4081,M03,2026-09-02 04:30:00,2026-09-01,Noche,0.9667,Diferencia de temperatura baja,Diferencia de temperatura baja (8.4 °C),Revisar refrigeración/ventilación


### ¿Qué señales aparecen más?

In [6]:
alertas_detalle["tipo_senal"].value_counts().rename("señales").to_frame()

,señales
tipo_senal,
Diferencia de temperatura baja,99
Sobreesfuerzo,95
Herramienta desgastada,79
Potencia alta,61
Potencia baja,29
Combinación de variables fuera de lo normal,14


### ¿Qué acciones hay que tomar? (alertas distintas por acción)
Se cuenta cada alerta **una sola vez por acción**, igual que hará `DISTINCTCOUNT` en Power BI.

In [7]:
(alertas_detalle.groupby("accion_sugerida")["id_registro"].nunique()
    .sort_values(ascending=False).rename("alertas").to_frame())

,alertas
accion_sugerida,
Revisar refrigeración/ventilación,99
Reducir carga o cambiar herramienta,95
Revisar parámetros de corte,90
Cambiar herramienta,79
Inspección general de la máquina,14


## 3. Verificaciones antes de guardar

Comprobamos que **el dashboard y los correos cuentan lo mismo** y que todas las llaves de las tablas de hechos
existen en sus dimensiones (si no, Power BI mostraría filas en *(En blanco)*).

In [8]:
total_correos = alertas_por_turno["cantidad_alertas"].sum()
total_detalle = alertas_detalle["id_registro"].nunique()
print(f"Alertas en los correos: {total_correos} · alertas en el detalle: {total_detalle} · "
      f"¿coinciden? {total_correos == total_detalle}")

hechos = {"oee_turnos": oee, "alertas_detalle": alertas_detalle,
          "alertas_por_turno": alertas_por_turno, "predicciones_fallas": predicciones}
filas = []
for nombre, tabla in hechos.items():
    filas.append({
        "tabla": nombre,
        "máquinas sin dimensión": len(set(tabla.get("id_maquina", [])) - set(dim_maquinas["id_maquina"])),
        "turnos sin dimensión": len(set(tabla["turno"]) - set(dim_turnos["turno"])),
        "fechas sin calendario": (len(set(pd.to_datetime(tabla["fecha_turno"])) - set(dim_fechas["fecha"]))
                                  if "fecha_turno" in tabla else "no tiene fecha_turno"),
    })
pd.DataFrame(filas)

Alertas en los correos: 284 · alertas en el detalle: 284 · ¿coinciden? True


,tabla,máquinas sin dimensión,turnos sin dimensión,fechas sin calendario
0,oee_turnos,0,0,0
1,alertas_detalle,0,0,0
2,alertas_por_turno,0,0,0
3,predicciones_fallas,0,0,0


## 4. Guardar los CSV

Se guardan con el contrato de datos del proyecto: **UTF-8, separador coma y fechas ISO 8601**.

In [9]:
salidas = {
    config.RUTA_CSV_DIM_MAQUINAS: dim_maquinas,
    config.RUTA_CSV_DIM_TURNOS: dim_turnos,
    config.RUTA_CSV_DIM_FECHAS: dim_fechas,
    config.RUTA_CSV_ALERTAS_DETALLE: alertas_detalle,
}
for ruta, tabla in salidas.items():
    guardar_csv_procesado(tabla, ruta)
    guardado = pd.read_csv(ruta)
    print(f"{ruta.name:<22} {guardado.shape[0]:>4} filas · columnas: {', '.join(guardado.columns)}")

dim_maquinas.csv          5 filas · columnas: id_maquina, nombre_maquina
dim_turnos.csv            3 filas · columnas: turno, orden_turno, hora_inicio, hora_fin, horario
dim_fechas.csv           43 filas · columnas: fecha, anio, mes_numero, mes_nombre, semana_iso, dia_semana_numero, dia_semana_nombre, es_fin_de_semana


alertas_detalle.csv     377 filas · columnas: id_registro, id_maquina, fecha_hora, fecha_turno, turno, probabilidad_falla, tipo_senal, senal, accion_sugerida


## Conclusión

- Las **3 dimensiones** (máquinas, turnos, calendario) permiten filtrar todo el dashboard con un solo control
  y ordenar turnos y meses correctamente.
- `alertas_detalle.csv` permite graficar **causas y acciones**; las alertas se cuentan con `DISTINCTCOUNT(id_registro)`.
- Las verificaciones confirman que el dashboard y los correos de n8n **cuentan las mismas alertas**.

**Siguiente paso:** cargar los CSV en Power BI, crear las relaciones y las medidas DAX (`powerbi/medidas_dax.md`).